In [1]:
# Import required libraries
import pandas as pd  # Main library for data manipulation and analysis
# Load the three datasets using pd.read_csv()
# This function reads CSV files and converts them into pandas DataFrames
customers = pd.read_csv('customers_wk9_thurs.csv')
products = pd.read_csv('products_wk9_thurs.csv')
sales = pd.read_csv('sales_wk9_thurs.csv')
# Display basic information about what we've loaded
# len() gives us the number of rows in each DataFrame
print(f"✅ Successfully loaded:")
print(f"   - {len(customers)} customers")      # Total number of customer records
print(f"   - {len(products)} products")        # Total number of products in catalog
print(f"   - {len(sales)} sales transactions") # Total number of sales transactions

# Display the first few rows of each dataset to understand the structure
# .head(n) shows the first n rows of a DataFrame - useful for data exploration
print(f"\n👀 First few rows of each dataset:")
print(f"\nCustomers:")
print(customers.head(3))  # Show first 3 rows of customer data
print(f"\nProducts:")
print(products.head(3))   # Show first 3 rows of product data
print(f"\nSales:")
print(sales.head(3))      # Show first 3 rows of sales data

✅ Successfully loaded:
   - 100 customers
   - 60 products
   - 150 sales transactions

👀 First few rows of each dataset:

Customers:
   customer_id first_name last_name  age state  income registration_date  \
0            1       John     Smith   34    CA   75000        2022-01-15   
1            2      Sarah   Johnson   28    NY   82000        2022-01-22   
2            3    Michael     Brown   45    TX   95000        2022-02-03   

                     email  
0     john.smith@email.com  
1  sarah.johnson@email.com  
2  michael.brown@email.com  

Products:
   product_id                   product_name       category  price  \
0           1  Wireless Bluetooth Headphones    Electronics  79.99   
1           2         Organic Cotton T-Shirt       Clothing  24.99   
2           3   Stainless Steel Water Bottle  Home & Garden  19.99   

  launch_date      brand  
0  2021-03-15  AudioTech  
1  2021-04-20    EcoWear  
2  2021-05-10  HydroLife  

Sales:
   sale_id  customer_id  product_id  

In [2]:
# =============================================================================
# PART 1: DATETIME OPERATIONS
# =============================================================================
# DateTime operations are crucial for time-based analysis in business data
# We'll convert text dates to proper datetime objects and extract useful components

# Convert date columns from strings to pandas datetime objects
# pd.to_datetime() is the main function for converting various date formats
# Once converted, we can perform date arithmetic and extract components
print("Converting date columns to datetime...")
customers['registration_date'] = pd.to_datetime(customers['registration_date'])
products['launch_date'] = pd.to_datetime(products['launch_date'])
sales['transaction_date'] = pd.to_datetime(sales['transaction_date'])
print("✅ All date columns converted successfully")

Converting date columns to datetime...
✅ All date columns converted successfully


In [3]:
# Extract useful components from datetime objects using the .dt accessor
# The .dt accessor gives us access to datetime-specific methods and properties
print("\nExtracting datetime components...")   
# Extract year (2022, 2023, etc.)
sales['year'] = sales['transaction_date'].dt.year
# Extract month number (1-12)
sales['month'] = sales['transaction_date'].dt.month
# Day of week (Monday=0, Sunday=6)
sales['day_of_week'] = sales['transaction_date'].dt.dayofweek

print("✅ Extracted: year, month, and day of week")


Extracting datetime components...
✅ Extracted: year, month, and day of week


In [4]:
# Find the date range of our sales data by using min() and max()
# This helps us understand the time span our analysis covers
earliest_date = sales['transaction_date'].min()
latest_date = sales['transaction_date'].max()

print("\n🔍 1.2 Basic Time Analysis")
print(f"Sales data covers {earliest_date} to {latest_date}")


🔍 1.2 Basic Time Analysis
Sales data covers 2022-02-10 14:30:00 to 2024-03-05 10:30:00


In [5]:
# Find the busiest day of the week by counting transactions
# .value_counts() counts occurrences of each unique value
# Count transactions for each day name
days_dict = {
    0: "Monday", 
    1: "Tuesday", 
    2: "Wednesday", 
    3: "Thursday",
    4: "Friday",
    5: "Saturday", 
    6: "Sunday"
}
day_counts = sales['day_of_week'].value_counts()
day_counts.index = day_counts.index.map(days_dict)
print(day_counts)

# Get the day with most transactions (first in sorted list)
busiest_day = day_counts.idxmax()
# Get the count for that day
busiest_count = day_counts.max()
# Busiest day as ranked by transaction count
print(f"\nBusiest day: {busiest_day} ({busiest_count} transactions)")

# Display all days ranked by transaction count for complete picture
parts = []
for i, day in enumerate(day_counts.index):
    parts.append(f"{i+1}) {day}")
ranking = ", ".join(parts)
print(f"\nDays by Transactions Ranking: {ranking}")

day_of_week
Friday       24
Tuesday      22
Thursday     21
Sunday       21
Saturday     21
Wednesday    21
Monday       20
Name: count, dtype: int64

Busiest day: Friday (24 transactions)

Days by Transactions Ranking: 1) Friday, 2) Tuesday, 3) Thursday, 4) Sunday, 5) Saturday, 6) Wednesday, 7) Monday


In [6]:
# Determine which month had the most transactions
month_dict = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

month_counts = sales['month'].value_counts()
month_counts.index = month_counts.index.map(month_dict)
print(month_counts)
print(f"\nMonth with most transactions: {month_counts.idxmax()} ({month_counts.max()} transactions)")

month
February     16
March        14
December     14
April        12
May          12
June         12
July         12
August       12
September    12
October      12
November     12
January      10
Name: count, dtype: int64

Month with most transactions: February (16 transactions)


In [7]:
# Count sales by year to see yearly trends
year_counts = sales['year'].value_counts().sort_index()

# Count transactions by year, sort chronologically
print(f"\nSales by year:\n{year_counts}")



Sales by year:
year
2022    65
2023    72
2024    13
Name: count, dtype: int64


In [8]:
# =============================================================================
# PART 2: GROUPBY AND AGGREGATION
# =============================================================================
# GroupBy operations allow us to split data into groups and perform calculations
# on each group separately. This is essential for analyzing patterns and trends.

In [9]:
# Group products by category and calculate average price for each category
# .groupby() splits the data into groups based on the 'category' column
# .mean() calculates the average for each group
# .round(2) rounds to 2 decimal places for currency formatting
# .sort_values(ascending=False) sorts from highest to lowest price
# Loop through each category and its average price             
# Format with dollar sign

category_stats = products.groupby('category')['price'].agg(avg_price='mean', product_count='count').sort_values('avg_price', ascending=False)
category_stats['avg_price'] = category_stats['avg_price'].round(2)
category_stats = category_stats.sort_values('avg_price', ascending=False)

for category, row in category_stats.iterrows():
    print(f"{category}: ${row['avg_price']:.2f} (n={int(row['product_count'])})")

Sports & Outdoors: $105.90 (n=11)
Electronics: $59.99 (n=13)
Clothing: $52.24 (n=12)
Accessories: $35.07 (n=12)
Home & Garden: $30.16 (n=12)


In [10]:
# Group customers by state and calculate average age
# This shows us the demographic profile of customers in different regions
customer_demographics = customers.groupby('state')['age'].mean().round(1)
customer_demographics

state
CA    34.1
FL    39.0
IL    37.2
NY    36.5
TX    38.3
Name: age, dtype: float64

In [11]:
# Calculate total spending per customer (sum up all their purchases)
total_spending_per_customer = sales.groupby('customer_id')['total_amount'].sum().round(2).sort_values(ascending=False)

for customer_id, total_amount in total_spending_per_customer.head(5).items():
    print(f"{customer_id}: ${total_amount}")

31: $379.97
13: $359.96
41: $329.98
84: $319.98
30: $284.98


In [12]:
# Find customers with the most purchases (transaction frequency)
# .groupby().size() counts the number of transactions per customer
# .size() is different from sum() - it counts rows, not values
# Customer ID with most purchases
# Number of purchases by top customer
# Show top 5 customers by number of purchases
purchase_counts = sales.groupby('customer_id').size().sort_values(ascending=False)

print("Distribution of purchase counts across customers:")
print(purchase_counts.value_counts().sort_index(ascending=False))

n_tied = (purchase_counts == purchase_counts.max()).sum()
print(f"\n{n_tied} customers are tied for the most purchases, at {purchase_counts.max()} transactions each")

Distribution of purchase counts across customers:
2    60
1    30
Name: count, dtype: int64

60 customers are tied for the most purchases, at 2 transactions each


In [13]:
# Calculate total sales revenue by month
# Group sales by month and sum the total_amount for each month
month_dict = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

total_sales_revenue_per_month = sales.groupby('month')['total_amount'].sum().round(2)
total_sales_revenue_per_month.index = total_sales_revenue_per_month.index.map(month_dict)

for month, total_amount in total_sales_revenue_per_month.items():
    print(f"{month}: ${total_amount:,.2f}")

January: $864.79
February: $1,669.66
March: $1,115.77
April: $924.77
May: $1,177.79
June: $986.68
July: $930.80
August: $870.74
September: $1,150.76
October: $1,235.80
November: $1,022.76
December: $1,458.71


In [14]:
# Calculate average transaction amount by day of week
# This shows us if people spend more on certain days
# Reorder by actual day order (Monday first) instead of alphabetical
days_dict = {
    0: "Monday", 
    1: "Tuesday", 
    2: "Wednesday", 
    3: "Thursday",
    4: "Friday",
    5: "Saturday", 
    6: "Sunday"
}

avg_transaction_amt_per_weekday = sales.groupby('day_of_week')['total_amount'].mean().round(2)
avg_transaction_amt_per_weekday.index = avg_transaction_amt_per_weekday.index.map(days_dict)
avg_transaction_amt_per_weekday

for day_of_week, total_amount in avg_transaction_amt_per_weekday.items():
    print(f"{day_of_week}: ${total_amount:,}")

Monday: $90.93
Tuesday: $74.03
Wednesday: $96.07
Thursday: $102.27
Friday: $100.94
Saturday: $61.12
Sunday: $99.55


In [15]:
# Which product category generates the most total revenue
sales_products = sales.merge(products, on='product_id')
revenue_by_category = sales_products.groupby('category')['total_amount'].sum().sort_values(ascending=False)

for category, total_amount in revenue_by_category.items():
    print(f"{category}: ${total_amount:,.2f}")

Sports & Outdoors: $3,544.64
Electronics: $3,524.29
Clothing: $2,745.39
Accessories: $2,053.25
Home & Garden: $1,541.46


In [16]:
# PART 3: Basic Data Insights
# =============================================================================
# Find which state has the most customers
state_counts = customers['state'].value_counts()
top_states = state_counts[state_counts == state_counts.max()].index.tolist()

print(f"State(s) with the most customers ({state_counts.max()} each):")
for state in top_states:
    print(state)

State(s) with the most customers (20 each):
CA
NY
TX
FL
IL


In [17]:
# Identify the top 5 customers by total spending
top_5_spenders = total_spending_per_customer.head(5).reset_index()
top_5_spenders = top_5_spenders.merge(customers[['customer_id', 'first_name', 'last_name']], on='customer_id')
top_5_spenders = top_5_spenders[['customer_id', 'first_name', 'last_name', 'total_amount']]
top_5_spenders

,customer_id,first_name,last_name,total_amount
0,31,Daniel,Gonzalez,379.97
1,13,Matthew,Harris,359.96
2,41,Joseph,Evans,329.98
3,84,Janice,Gonzales,319.98
4,30,Kimberly,Baker,284.98


In [18]:
# Determine which day of the week has the highest average transaction value
best_weekday = avg_transaction_amt_per_weekday.idxmax()
print(f"\nHighest avg transaction value: {best_weekday} ${avg_transaction_amt_per_weekday.max():.2f}")


Highest avg transaction value: Thursday $102.27


In [19]:
# Discussion Question 1: DateTime Benefits
# How can extracting day of week and month from dates help a retail business make better decisions?

# Extracting weekday and month turns a series of raw dates into designated categories you can summmarize and/or analyze.  For example, weekday data allows for better near-term decision-making on weekly staffing and inventory
# Whereas monthly data can not only look at a larger span of time, but also provides a lens into annual, seasonal, or monthly patterns that can be better inform long-strategy or at least investigations into how business
# model interacts with the overall 12- month calendar

In [20]:
# Discussion Question 2: GroupBy Value
# Why is it useful to group data by categories, states, or time periods rather than looking at individual transactions?

# Individual transactions are often too numerous in quantity and affected by too many counfounding/attenuating factors to be useful in making conclusions on brought together, if each has to be taken into account  
# Grouping meanwhile collapses individual transactions variables into summations at that specific level that allow for some level of comparison given that the rest of variable information is/should be relatively uniform
# in how it was input into table

In [21]:
# Discussion Question 3: Business Applications
# Based on your findings, what recommendations would you give to help the retail chain improve their performance?

# Recommendation 1: Friday = highest volume (24 txns), Thursday = highest avg value ($102.27 vs. low of $61.12 Saturday) -> staff for Friday, run upsell/premium promos on Thursday
# Recommendation 2: February = seasonal peak (16 txns, aggregated across 2022-2024) -> front-load inventory and campaigns ahead of it
# Recommendation 3: Sports & Outdoors = top revenue ($3,544.64) with fewest SKUs (11); Home & Garden = lowest revenue ($1,541.46) despite more SKUs (12) -> expand Sports & Outdoors, cut Home & Garden shelf space
# Recommendation 4: Purchase frequency caps at 2 (60 customers) or 1 (30 customers) - no repeat-buyer whales -> treat as an acquisition/conversion problem, not a VIP-retention play